# AGML-DenseCBAM — Leakage-Safe Training

This notebook is a thin interface to the active Python pipeline. Keeping model and evaluation logic in one implementation prevents the notebook and scripts from drifting apart.

Final workflow: audit data → generate leakage-safe folds → smoke test → run C1–C4 → aggregate Chapter IV results.

In [ ]:
import subprocess
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'scripts' / 'train_one_case_5fold.py').exists():
            return candidate
    raise RuntimeError('Could not locate the AGML-DenseCBAM project root.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SCRIPTS_ROOT = PROJECT_ROOT / 'scripts'
DATA_ROOT = PROJECT_ROOT / 'data'
FOLDS_ROOT = PROJECT_ROOT / 'data_5_fold'
RESULTS_ROOT = PROJECT_ROOT / 'chapter4_results' / 'final_v2'

# Safety switches: explicitly enable only the stages you intend to run.
REGENERATE_FOLDS = False  # set True only after exclusion approval
RUN_SMOKE_TESTS = False
RUN_FINAL_CASES = False   # enable only after the V2 protocol is locked
RUN_ANALYSIS = False
SKIP_EXISTING = True      # safely resumes only matching code/config/folds

print('Python:', sys.version)
print('Executable:', sys.executable)
print('Project:', PROJECT_ROOT)
print('Final results:', RESULTS_ROOT)

## 1. Audit and generate folds

The default `error` policy writes `duplicate_audit.csv` and stops when identical pixels have conflicting labels. Review those rows with the dataset owner. Change to `exclude` only when exclusion has been formally approved and will be documented in the thesis. Add `--groups_csv ...` when patient/study IDs are available.

In [ ]:
CONFLICT_POLICY = 'error'  # change to 'exclude' only after review/approval
if REGENERATE_FOLDS:
    fold_command = [
        sys.executable, str(SCRIPTS_ROOT / 'make_5fold_dataset.py'),
        '--input_root', str(DATA_ROOT),
        '--output_root', str(FOLDS_ROOT),
        '--n_splits', '5', '--val_size', '0.15', '--seed', '42',
        '--conflict_policy', CONFLICT_POLICY,
    ]
    subprocess.run(fold_command, cwd=PROJECT_ROOT, check=True)
else:
    print('Fold regeneration skipped; using:', FOLDS_ROOT)

## 2. Smoke test

This uses one fold and five epochs in a separate V2 output directory. It is a development check, not a reportable result.

In [ ]:
SMOKE_CASES = [('benchmark', 'artifact_mix'), ('proposed', 'artifact_mix')]
if RUN_SMOKE_TESTS:
    for model_type, scenario in SMOKE_CASES:
        smoke_command = [
            sys.executable, str(SCRIPTS_ROOT / 'run_case_5fold_isolated.py'),
            '--folds_root', str(FOLDS_ROOT),
            '--model_type', model_type, '--scenario', scenario,
            '--epochs', '5', '--batch_size', '8', '--fold_limit', '1',
            '--artifact_loss_weight', '0.1',
            '--output_dir', str(PROJECT_ROOT / 'chapter4_results' / 'smoke_v2' / f'{model_type}_{scenario}'),
        ]
        if SKIP_EXISTING:
            smoke_command.append('--skip_existing')
        print('\nSMOKE TEST:', model_type, scenario)
        subprocess.run(smoke_command, cwd=PROJECT_ROOT, check=True)
else:
    print('Smoke tests skipped.')

## 3. Final C1–C4 runs

Run all cases on the same folds, seed, hyperparameters, hardware, and locked V2 artifact protocol. A fresh process is used for each fold. This can take many hours.

In [ ]:
EPOCHS = 50
CASES = [
    ('benchmark', 'clean'),
    ('benchmark', 'artifact_mix'),
    ('proposed', 'clean'),
    ('proposed', 'artifact_mix'),
]

if RUN_FINAL_CASES:
    for model_type, scenario in CASES:
        output_dir = RESULTS_ROOT / f'{model_type}_{scenario}'
        command = [
            sys.executable, str(SCRIPTS_ROOT / 'run_case_5fold_isolated.py'),
            '--folds_root', str(FOLDS_ROOT),
            '--model_type', model_type, '--scenario', scenario,
            '--epochs', str(EPOCHS), '--batch_size', '8',
            '--learning_rate', '1e-4', '--l2_strength', '1e-2',
            '--artifact_loss_weight', '0.1',
            '--seed', '42', '--output_dir', str(output_dir),
        ]
        if SKIP_EXISTING:
            command.append('--skip_existing')
        print('\nFINAL CASE:', model_type, scenario)
        subprocess.run(command, cwd=PROJECT_ROOT, check=True)
else:
    print('Final C1-C4 runs are disabled. Set RUN_FINAL_CASES = True after smoke tests pass.')

## 4. Chapter IV aggregate analysis

The analyzer refuses incomplete case sets. Outputs include mean ± SD, 95% CIs, paired tests, robustness degradation comparisons, and graphs.

In [ ]:
if RUN_ANALYSIS:
    subprocess.run([
        sys.executable, str(SCRIPTS_ROOT / 'analyze_chapter4.py'),
        '--results_root', str(RESULTS_ROOT),
        '--expected_folds', '5',
    ], cwd=PROJECT_ROOT, check=True)
else:
    print('Analysis skipped. Enable RUN_ANALYSIS after all four final cases complete.')

## Interpretation safeguards

- Do not report `oldstyle_results` or smoke-test metrics as final results.
- State whether patient/study grouping was available. Hash deduplication alone does not establish patient independence.
- Describe artifacts as synthetic corruptions, not verified real clinical artifact classes.
- Five-fold inferential tests have low power; report fold values, confidence intervals, and effect sizes with p-values.
- Grad-CAM remains qualitative unless expert ROI annotations are available.